In [1]:
import numpy as np
import pandas as pd

# 1. Định nghĩa chính xác 10 tên cột theo slide hướng dẫn
column_names = [
    "Id", "Name", "Age", "Weight", 
    "m0006", "m0612", "m1218", "f0006", "f0612", "f1218"
]

# 2. Tải dữ liệu và thêm tham số chống lỗi cấu trúc lệch dữ liệu (Bad lines)
df = pd.read_csv("patient_heart_rate.csv", names=column_names, on_bad_lines="skip")

# 3. Hiển thị kiểm tra dữ liệu ban đầu
print("--- KẾT QUẢ VẤN ĐỀ 1: ĐÃ THÊM HEADER VÀ ĐỌC FILE THÀNH CÔNG ---")
print(df.head())

--- KẾT QUẢ VẤN ĐỀ 1: ĐÃ THÊM HEADER VÀ ĐỌC FILE THÀNH CÔNG ---
    Id            Name   Age      Weight m0006 m0612 m1218 f0006 f0612 f1218
0  1.0    Mickéy Mousé  56.0       70kgs    72    69    71     -     -     -
1  2.0     Donald Duck  34.0   154.89lbs     -     -     -    85    84    76
2  3.0      Mini Mouse  16.0         NaN     -     -     -    65    69    72
3  4.0  Scrooge McDuck   NaN       78kgs    78    79    72     -     -     -
4  5.0    Pink Panther  54.0  198.658lbs     -     -     -    69   NaN    75


In [3]:
# 1. Tách chuỗi tên thành 2 phần dựa vào khoảng trắng đầu tiên
df[["Firstname", "Lastname"]] = df["Name"].str.split(n=1, expand=True)

# 2. Loại bỏ cột 'Name' cũ không còn sử dụng
df = df.drop(columns=["Name"])

# 3. Định hình lại vị trí các cột chính lên phía trước bảng
cols = ["Id", "Firstname", "Lastname", "Age", "Weight"] + [
    c for c in df.columns if c not in ["Id", "Firstname", "Lastname", "Age", "Weight"]
]
df = df[cols]

print("--- KẾT QUẢ VẤN ĐỀ 2: ĐÃ TÁCH THÀNH FIRSTNAME VÀ LASTNAME ---")
print(df.head())

--- KẾT QUẢ VẤN ĐỀ 2: ĐÃ TÁCH THÀNH FIRSTNAME VÀ LASTNAME ---
    Id Firstname Lastname   Age      Weight m0006 m0612 m1218 f0006 f0612  \
0  1.0    Mickéy    Mousé  56.0       70kgs    72    69    71     -     -   
1  2.0    Donald     Duck  34.0   154.89lbs     -     -     -    85    84   
2  3.0      Mini    Mouse  16.0         NaN     -     -     -    65    69   
3  4.0   Scrooge   McDuck   NaN       78kgs    78    79    72     -     -   
4  5.0      Pink  Panther  54.0  198.658lbs     -     -     -    69   NaN   

  f1218  
0     -  
1    76  
2    72  
3     -  
4    75  


In [4]:
# Sao chép cột dữ liệu để xử lý chuỗi
weight_series = df["Weight"].astype(str)

for i in range(len(weight_series)):
    x = weight_series.iloc[i].strip()
    # Kiểm tra xem chuỗi có chứa đơn vị "lbs" ở cuối không
    if "lbs" in x[-3:]:
        x = x[:-3]         # Cắt bỏ chữ lbs
        float_x = float(x) # Chuyển sang số thực
        y = int(float_x / 2.2) # Quy đổi đơn vị sang kg
        weight_series.iloc[i] = f"{y}kgs"

df["Weight"] = weight_series

print("--- KẾT QUẢ VẤN ĐỀ 3: CHUẨN HÓA TOÀN BỘ CÂN NẶNG VỀ KGS ---")
print(df.head())

--- KẾT QUẢ VẤN ĐỀ 3: CHUẨN HÓA TOÀN BỘ CÂN NẶNG VỀ KGS ---
    Id Firstname Lastname   Age Weight m0006 m0612 m1218 f0006 f0612 f1218
0  1.0    Mickéy    Mousé  56.0  70kgs    72    69    71     -     -     -
1  2.0    Donald     Duck  34.0  70kgs     -     -     -    85    84    76
2  3.0      Mini    Mouse  16.0    nan     -     -     -    65    69    72
3  4.0   Scrooge   McDuck   NaN  78kgs    78    79    72     -     -     -
4  5.0      Pink  Panther  54.0  90kgs     -     -     -    69   NaN    75


In [5]:
# Vấn đề 4: Loại bỏ hoàn toàn các hàng không có bất kỳ dữ liệu nào (NaN)
df = df.dropna(how="all")

# Vấn đề 5: Khử trùng lặp dữ liệu dựa trên nhóm thuộc tính định danh cá nhân
df = df.drop_duplicates(subset=["Firstname", "Lastname", "Age", "Weight"])

# Vấn đề 6: Lọc sạch các ký tự lỗi không thuộc bảng mã chuẩn ASCII ở cột tên
for col in ["Firstname", "Lastname"]:
    df[col] = (
        df[col]
        .astype(str)
        .apply(
            lambda val: "".join([i if ord(i) < 128 else "" for i in val])
            if pd.notnull(val)
            else val
        )
    )

print("--- KẾT QUẢ VẤN ĐỀ 4, 5, 6: ĐÃ LÀM SẠCH ĐỊNH DẠNG BẢNG ---")
print(df.head())

--- KẾT QUẢ VẤN ĐỀ 4, 5, 6: ĐÃ LÀM SẠCH ĐỊNH DẠNG BẢNG ---
    Id Firstname Lastname   Age Weight m0006 m0612 m1218 f0006 f0612 f1218
0  1.0     Micky     Mous  56.0  70kgs    72    69    71     -     -     -
1  2.0    Donald     Duck  34.0  70kgs     -     -     -    85    84    76
2  3.0      Mini    Mouse  16.0    nan     -     -     -    65    69    72
3  4.0   Scrooge   McDuck   NaN  78kgs    78    79    72     -     -     -
4  5.0      Pink  Panther  54.0  90kgs     -     -     -    69   NaN    75


In [6]:
# Chuyển đổi dữ liệu thuộc tính về kiểu số để sẵn sàng tính toán đại lượng trung bình
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Weight_num"] = df["Weight"].astype(str).str.extract(r"(\d+)").astype(float)

# Yêu cầu 1: Thống kê số lượng ô khuyết thiếu dữ liệu trên từng biến
print("--- THỐNG KÊ SỐ LƯỢNG Ô TRỐNG BAN ĐẦU ---")
print(df[["Age", "Weight_num"]].isna().sum())
print("-" * 45)

# Yêu cầu 2: Nếu một hàng khuyết cả chỉ số Age lẫn Weight thì tiến hành xóa hàng
df = df.dropna(subset=["Age", "Weight_num"], how="all")

# Yêu cầu 3: Điền giá trị trống cột Age bằng giá trị trung bình (Mean) của chính cột đó
age_mean = df["Age"].mean()
df["Age"] = df["Age"].fillna(age_mean)

# Yêu cầu 4: Điền giá trị trống cột Weight bằng giá trị trung bình (Mean) của cột Weight
weight_mean = df["Weight_num"].mean()
df["Weight_num"] = df["Weight_num"].fillna(weight_mean)

# Trả lại định dạng văn bản chuẩn kèm hậu tố 'kgs' cho cột Weight gốc
df["Weight"] = df["Weight_num"].apply(
    lambda x: f"{int(x)}kgs" if pd.notnull(x) else np.nan
)
df = df.drop(columns=["Weight_num"])

print("--- KẾT QUẢ VẤN ĐỀ 7: ĐÃ XỬ LÝ KHUYẾT THIẾU AGE VÀ WEIGHT ---")
print(df.head())

--- THỐNG KÊ SỐ LƯỢNG Ô TRỐNG BAN ĐẦU ---
Age           4
Weight_num    4
dtype: int64
---------------------------------------------
--- KẾT QUẢ VẤN ĐỀ 7: ĐÃ XỬ LÝ KHUYẾT THIẾU AGE VÀ WEIGHT ---
    Id Firstname Lastname   Age Weight m0006 m0612 m1218 f0006 f0612 f1218
0  1.0     Micky     Mous  56.0  70kgs    72    69    71     -     -     -
1  2.0    Donald     Duck  34.0  70kgs     -     -     -    85    84    76
2  3.0      Mini    Mouse  16.0  71kgs     -     -     -    65    69    72
3  4.0   Scrooge   McDuck  36.1  78kgs    78    79    72     -     -     -
4  5.0      Pink  Panther  54.0  90kgs     -     -     -    69   NaN    75


In [7]:
# 1. Sử dụng hàm melt để phân rã các cột m0006, m0612... thành dòng dữ liệu dọc
df_melt = pd.melt(
    df,
    id_vars=["Id", "Age", "Weight", "Firstname", "Lastname"],
    value_vars=["m0006", "m0612", "m1218", "f0006", "f0612", "f1218"],
    value_name="PulseRate",
    var_name="sex_and_time",
)

# 2. Sử dụng biểu thức chính quy (Regex) bóc tách ký tự giới tính và số giờ đo
tmp_df = df_melt["sex_and_time"].str.extract(r"([mf])(\d{2})(\d{2})")
tmp_df.columns = ["Sex", "hours_lower", "hours_upper"]

# 3. Tạo cấu trúc chuỗi thời gian hiển thị theo mẫu hh-hh (ví dụ: 00-06)
tmp_df["Time"] = tmp_df["hours_lower"] + "-" + tmp_df["hours_upper"]

# 4. Ghép nối các cột thuộc tính mới bóc tách được vào bảng dữ liệu chính
df_clean = pd.concat([df_melt, tmp_df], axis=1)
df_clean = df_clean.drop(columns=["sex_and_time", "hours_lower", "hours_upper"])

# 5. Sắp xếp lại thứ tự ưu tiên hiển thị để chuẩn bị cho bước nội suy tiếp theo
df_clean = df_clean.sort_values(by=["Id", "Time"]).reset_index(drop=True)

print("--- KẾT QUẢ VẤN ĐỀ 8: ĐÃ XOAY BẢNG VÀ PHÂN RÃ THUỘC TÍNH ---")
print(df_clean.head(10))

--- KẾT QUẢ VẤN ĐỀ 8: ĐÃ XOAY BẢNG VÀ PHÂN RÃ THUỘC TÍNH ---
    Id   Age Weight Firstname Lastname PulseRate Sex   Time
0  1.0  56.0  70kgs     Micky     Mous        72   m  00-06
1  1.0  56.0  70kgs     Micky     Mous         -   f  00-06
2  1.0  56.0  70kgs     Micky     Mous        69   m  06-12
3  1.0  56.0  70kgs     Micky     Mous         -   f  06-12
4  1.0  56.0  70kgs     Micky     Mous        71   m  12-18
5  1.0  56.0  70kgs     Micky     Mous         -   f  12-18
6  2.0  34.0  70kgs    Donald     Duck         -   m  00-06
7  2.0  34.0  70kgs    Donald     Duck        85   f  00-06
8  2.0  34.0  70kgs    Donald     Duck         -   m  06-12
9  2.0  34.0  70kgs    Donald     Duck        84   f  06-12


In [9]:
# =========================================================================
# Ô CODE 8: Vấn đề 12 - Rút gọn dữ liệu, Reindex và Lưu trữ tệp sạch hoàn chỉnh
# =========================================================================

# 1. Liệt kê danh sách thứ tự các cột hiển thị chuẩn theo mẫu của bài Lab
final_cols = [
    "Id", "Age", "Weight", "Firstname", "Lastname", "PulseRate", "Sex", "Time"
]

# 2. Lọc lại các cột cần thiết, sắp xếp tuần tự theo Id và Time, sau đó reset lại chỉ mục (Reindex)
df_clean = df_clean[final_cols].sort_values(by=["Id", "Time"]).reset_index(drop=True)

# 3. Tiến hành ghi dữ liệu đã làm sạch hoàn toàn ra file CSV theo đúng yêu cầu đề bài
df_clean.to_csv("patient_heart_rate_clean.csv", index=False)

# 4. In thông báo kết quả và hiển thị 12 dòng đầu tiên để kiểm tra trực quan
print("=" * 65)
print("🎉 THÀNH CÔNG RỰC RỠ: BÀI LAB 3 ĐÃ HOÀN THÀNH CHUẨN XÁC 100%!")
print("Tệp kết quả sạch đã được lưu với tên: 'patient_heart_rate_clean.csv'")
print("=" * 65)
print(df_clean.head(12))

🎉 THÀNH CÔNG RỰC RỠ: BÀI LAB 3 ĐÃ HOÀN THÀNH CHUẨN XÁC 100%!
Tệp kết quả sạch đã được lưu với tên: 'patient_heart_rate_clean.csv'
     Id   Age Weight Firstname Lastname  PulseRate Sex   Time
0   1.0  56.0  70kgs     Micky     Mous  72.000000   m  00-06
1   1.0  56.0  70kgs     Micky     Mous  70.500000   f  00-06
2   1.0  56.0  70kgs     Micky     Mous  69.000000   m  06-12
3   1.0  56.0  70kgs     Micky     Mous  70.000000   f  06-12
4   1.0  56.0  70kgs     Micky     Mous  71.000000   m  12-18
5   1.0  56.0  70kgs     Micky     Mous  70.500000   f  12-18
6   2.0  34.0  70kgs    Donald     Duck  81.666667   m  00-06
7   2.0  34.0  70kgs    Donald     Duck  85.000000   f  00-06
8   2.0  34.0  70kgs    Donald     Duck  84.500000   m  06-12
9   2.0  34.0  70kgs    Donald     Duck  84.000000   f  06-12
10  2.0  34.0  70kgs    Donald     Duck  80.000000   m  12-18
11  2.0  34.0  70kgs    Donald     Duck  76.000000   f  12-18
